# DiagramIO: đo hiệu năng pha 2

Notebook chạy bộ đo trong thư mục `bench/` của repo DiagramIO (xem `bench/README.md` và `docs/adr/0001-phuong-phap-do.md`).

- **Settings**: Accelerator **None** (CPU), bật **Internet**.
- Chạy nền để không phụ thuộc trình duyệt: **Save Version → Save & Run All (Commit)**. Nên chạy `quick` trước (khoảng 15 phút) để kiểm tra môi trường.
- Ma trận `b12`: C0 (`v2.0-m0`), C1 (B1, `v2.1-b1`), C2 (B2, cache Caffeine), C3 (B2, Caffeine + Redis), chạy xen kẽ trong cùng phiên; khoảng 8-9 giờ với 3 lần lặp, đặt `REPS = 2` nếu cần nhanh hơn (khoảng 6 giờ). Ma trận `tune`: tinh chỉnh runtime của B1 (khoảng 1,5 giờ). Ma trận `m0`: chỉ C0.
- Kết quả: thư mục `DiagramIO/bench/results/<phiên>/` và file `results.tgz` trong tab **Output**. Giải nén vào `bench/results/` của repo rồi commit các file tóm tắt.

In [ ]:
REPO_URL = "https://github.com/vietsosad/DiagramIO_16.git"
REF = "main"      # commit hoặc tag chứa thư mục bench/; cấu hình C2, C3 của b12 được build từ REF
MATRIX = "b12"    # "quick" để chạy thử, "tune" để tinh chỉnh runtime, "m0" chỉ đo C0
REPS = 3          # số lần lặp mỗi phép đo (phương pháp: 3)

In [ ]:
import os
import subprocess

os.chdir("/kaggle/working")
if not os.path.isdir("DiagramIO"):
    subprocess.run(["git", "clone", REPO_URL, "DiagramIO"], check=True)
os.chdir("/kaggle/working/DiagramIO")
subprocess.run(["git", "fetch", "--tags", "origin"], check=True)
subprocess.run(["git", "checkout", "--detach", REF], check=True)
print(subprocess.run(["git", "log", "-1", "--format=%H %s"], capture_output=True, text=True).stdout)

In [ ]:
# Cài PostgreSQL 16, Redis 7.4, JDK 21, k6, async-profiler và cấu hình PostgreSQL (khoảng 3-4 phút)
!bash bench/kaggle/setup.sh

In [ ]:
# Build jar từng cấu hình (C0, C1 từ tag; C2, C3 từ REF), sinh và nạp dữ liệu, EXPLAIN, chạy cả ma trận, báo cáo
!REPS={REPS} REF_C2={REF} REF_C3={REF} bash bench/kaggle/run_matrix.sh {MATRIX}

In [ ]:
# Đóng gói toàn bộ kết quả (kể cả log thô) để tải về từ tab Output
!cd bench/results && tar czf /kaggle/working/results.tgz . && ls -lh /kaggle/working/results.tgz

In [ ]:
import glob
from IPython.display import Markdown, display

latest = sorted(glob.glob("bench/results/*/report.md"))[-1]
print(latest)
display(Markdown(open(latest, encoding="utf-8").read()))